<h1>Orientador - Clase 10</h1>
<h4>Actividad 1</h4>
<p><b>a) ¿Qué significa que la información persista en el tiempo?</b></p>
<p>Significa que la información guardada se conserva más allá de la ejecución del programa que la creó. Los datos siguen existiendo aunque el programa termine o se apague la computadora.</p>
<p><b>b) ¿Qué diferencias encuentran entre una variable en memoria y un archivo?</b></p>
<p>Una variable vive en la memoria RAM mientras el programa se ejecuta: cuando el programa termina, su valor se pierde. Un archivo se guarda en un dispositivo de almacenamiento (disco), por lo que sigue existiendo con la información guardada aunque el programa haya terminado.</p>
<p><b>c) ¿Qué ventajas y limitaciones presenta almacenar los datos como texto?</b></p>
<p><b>Ventajas:</b> se puede leer con cualquier editor sin programas externos y es fácil detectar errores mirando el archivo.</p>
<p><b>Limitaciones:</b> ocupa más espacio que un archivo binario, hay que convertir los datos (de str a int o float) al leerlos, un dato con el delimitador (por ejemplo ";") rompe el registro, y las búsquedas son lentas porque hay que recorrer el archivo línea por línea.</p>

<h4>Actividad 2</h4>
<p><b>a) "wt": escritura; crea el archivo o reemplaza todo su contenido si ya existe.</b></p>

In [1]:
with open("notas.txt", "wt", encoding="utf-8") as archivo:
    archivo.write("Primer registro\n")
    archivo.write("Segundo registro\n")
    archivo.write("Tercer registro\n")

<p><b>b) "at": agregado; crea el archivo o escribe al final sin borrar lo anterior.</b></p>

In [2]:
with open("notas.txt", "at", encoding="utf-8") as archivo:
    archivo.write("Cuarto registro\n")

<p><b>c) "rt": lectura; el archivo debe existir.</b></p>

In [3]:
with open("notas.txt", "rt", encoding="utf-8") as archivo:
    print(archivo.read())

Primer registro
Segundo registro
Tercer registro
Cuarto registro



<p>La función <code>open()</code> abre el archivo. Recibe el nombre, el modo (lectura, escritura, agregado) y la codificación con la que se traducen los caracteres a bytes (utf-8).</p>
<p>La función <code>write()</code> graba una cadena en el archivo. Solo funciona en los modos <b>"wt"</b> y <b>"at"</b>: en modo <b>"rt"</b> da error. No agrega el salto de línea solo, hay que escribir <code>"\n"</code> en cada registro.</p>
<p><code>with</code> garantiza que el archivo se cierre siempre, incluso si ocurre un error dentro del bloque. Sin <code>with</code> habría que llamar a <code>archivo.close()</code> al terminar.</p>

<h4>Actividad 3</h4>
<p>Cada alumno se guarda en una línea, con los campos separados por punto y coma. Se valida que el legajo sea positivo, que nombre y apellido no estén vacíos y que la comisión sea un número de 1 a 10. La carga termina con el legajo -1, que no se graba.</p>

In [4]:
def pedir_legajo():
    while True:
        try:
            legajo = int(input("Legajo (-1 para finalizar): "))
        except ValueError:
            print("El legajo debe ser un número entero.")
        else:
            if legajo == -1 or legajo > 0:
                return legajo
            print("El legajo debe ser positivo.")


def pedir_texto(dato):
    valor = input(f"Ingrese el {dato}: ").strip()
    while valor == "" or ";" in valor:
        print(f"El {dato} está vacío o contiene ';'.")
        valor = input(f"Ingrese el {dato}: ").strip()
    return valor


def pedir_comision():
    while True:
        try:
            comision = int(input("Ingrese la comisión (1 a 10): "))
        except ValueError:
            print("La comisión debe ser un número.")
        else:
            if 1 <= comision <= 10:
                return comision
            print("La comisión debe estar entre 1 y 10.")


def crear_alumnos(modo="at"):
    try:
        with open("alumnos.txt", modo, encoding="utf-8") as archivo:
            legajo = pedir_legajo()
            while legajo != -1:
                nombre = pedir_texto("nombre")
                apellido = pedir_texto("apellido")
                comision = pedir_comision()
                archivo.write(f"{legajo};{nombre};{apellido};{comision}\n")
                legajo = pedir_legajo()
    except OSError as error:
        print("No se pudo escribir el archivo:", error)

<p><b>Prueba con modo "wt":</b></p>

In [5]:
crear_alumnos("wt")

<p><b>Prueba con modo "at":</b> cargamos más alumnos sobre el mismo archivo.</p>

In [6]:
crear_alumnos("at")

<p>Con el modo <b>"wt"</b> los datos anteriores se pierden: el archivo se vacía y solo quedan los alumnos de la nueva carga. Con el modo <b>"at"</b> los datos anteriores se conservan y los nuevos se agregan al final del archivo.</p>

<p>Agrego tres registros mal formados a propósito, para probar que la lectura los informa y los omite.</p>

In [7]:
with open("alumnos.txt", "at", encoding="utf-8") as archivo:
    archivo.write("10200;Julia;Paz\n")
    archivo.write("abc;Pedro;López;4\n")
    archivo.write("5000;Lucía;Gómez;4;extra\n")

<h4>Actividad 4</h4>
<p>Se lee el archivo registro por registro, sin cargarlo completo en memoria. Con <code>rstrip("\n")</code> se saca solo el salto de línea final (<code>strip()</code> también borraría espacios al principio y al final). Antes de convertir el legajo se verifica que la línea tenga exactamente cuatro campos.</p>

In [8]:
def leer_alumno(linea):
    """Devuelve (legajo, nombre, apellido, comision) o lanza ValueError."""
    campos = linea.rstrip("\n").split(";")
    if len(campos) != 4:
        raise ValueError(f"cantidad de campos incorrecta ({len(campos)})")
    legajo = int(campos[0])
    comision = int(campos[3])
    return legajo, campos[1], campos[2], comision

<p><b>a) Mostrar todos los registros válidos</b> (d: las líneas inválidas se informan y se omiten).</p>

In [9]:
def mostrar_alumnos():
    try:
        with open("alumnos.txt", "rt", encoding="utf-8") as archivo:
            numero = 0
            for linea in archivo:
                numero += 1
                try:
                    legajo, nombre, apellido, comision = leer_alumno(linea)
                except ValueError as error:
                    print(f"Línea {numero} inválida: {error}")
                    continue
                print(f"Legajo: {legajo} | {nombre} {apellido} | Comisión: {comision}")
    except OSError as error:
        print("No se pudo leer el archivo:", error)

mostrar_alumnos()

Legajo: 909 | Ana Gomez | Comisión: 10
Legajo: 10020 | Luis Fernandez | Comisión: 3
Legajo: 910 | Marta Rodriguez | Comisión: 2
Legajo: 10050 | Sofia Martinez | Comisión: 10
Legajo: 10100 | Valentina Diaz | Comisión: 3
Línea 6 inválida: cantidad de campos incorrecta (3)
Línea 7 inválida: invalid literal for int() with base 10: 'abc'
Línea 8 inválida: cantidad de campos incorrecta (5)


<p><b>b) Informar solo los alumnos con legajo mayor a 10000</b></p>

In [10]:
def mostrar_mayores(minimo):
    try:
        with open("alumnos.txt", "rt", encoding="utf-8") as archivo:
            for linea in archivo:
                try:
                    legajo, nombre, apellido, comision = leer_alumno(linea)
                except ValueError:
                    continue   # las inválidas ya se informan en el punto a)
                if legajo > minimo:
                    print(f"Legajo: {legajo} | {nombre} {apellido}")
    except OSError as error:
        print("No se pudo leer el archivo:", error)

mostrar_mayores(10000)

Legajo: 10020 | Luis Fernandez
Legajo: 10050 | Sofia Martinez
Legajo: 10100 | Valentina Diaz


<p><b>c) Contar cuántos alumnos hay en cada comisión con un diccionario</b></p>

In [11]:
def contar_por_comision():
    conteo = {}
    try:
        with open("alumnos.txt", "rt", encoding="utf-8") as archivo:
            for linea in archivo:
                try:
                    legajo, nombre, apellido, comision = leer_alumno(linea)
                except ValueError:
                    continue
                conteo[comision] = conteo.get(comision, 0) + 1
    except OSError as error:
        print("No se pudo leer el archivo:", error)
        return

    for comision in sorted(conteo):
        print(f"Comisión {comision}: {conteo[comision]} alumno(s)")

contar_por_comision()

Comisión 2: 1 alumno(s)
Comisión 3: 2 alumno(s)
Comisión 10: 2 alumno(s)


<p><code>conteo.get(comision, 0)</code> devuelve 0 si la comisión todavía no estaba en el diccionario, así que no hace falta preguntar antes si existe la clave.</p>

<h4>Actividad 5</h4>
<p>Un archivo CSV sigue siendo un archivo de texto, pero con estructura de tabla. Acá el delimitador es ";" y el archivo empieza con un encabezado: <code>codigo;descripcion;cantidad;precio</code>. No se usa el módulo <code>csv</code>: todo se resuelve con <code>split()</code> y <code>write()</code>.</p>

In [12]:
ENCABEZADO_PROD = "codigo;descripcion;cantidad;precio"


def cargar_productos():
    try:
        with open("productos.csv", "wt", encoding="utf-8") as archivo:
            archivo.write(ENCABEZADO_PROD + "\n")
            codigo = input("Código (FIN para terminar): ").strip()
            while codigo.upper() != "FIN":
                descripcion = input("Descripción: ").strip()
                try:
                    cantidad = int(input("Cantidad: "))
                    precio = float(input("Precio: ").replace(",", "."))
                except ValueError:
                    print("Cantidad o precio inválidos.")
                else:
                    if codigo == "" or descripcion == "" or ";" in codigo or ";" in descripcion:
                        print("Código o descripción vacíos o con ';'.")
                    elif cantidad < 0:
                        print("La cantidad no puede ser negativa.")
                    elif precio <= 0:
                        print("El precio debe ser mayor que cero.")
                    else:
                        archivo.write(f"{codigo};{descripcion};{cantidad};{precio}\n")
                codigo = input("Código (FIN para terminar): ").strip()
    except OSError as error:
        print("No se pudo escribir el archivo:", error)

In [13]:
cargar_productos()

<p>El valor <b>FIN</b> no se guarda porque el <code>while</code> termina antes del <code>write()</code>. Al leer, el encabezado se lee aparte con <code>readline()</code> y se verifica, para no procesarlo como si fuera un producto.</p>

In [14]:
def leer_productos():
    total = 0
    pocos = []
    try:
        with open("productos.csv", "rt", encoding="utf-8") as archivo:
            if archivo.readline().rstrip("\n") != ENCABEZADO_PROD:
                print("El archivo no tiene el encabezado esperado.")
                return

            numero = 1
            for linea in archivo:
                numero += 1
                campos = linea.rstrip("\n").split(";")

                if len(campos) != 4:
                    print(f"Línea {numero} inválida: cantidad de campos incorrecta.")
                    continue

                try:
                    cantidad = int(campos[2])
                    precio = float(campos[3])
                except ValueError:
                    print(f"Línea {numero} inválida: cantidad o precio no numéricos.")
                    continue

                if campos[0] == "" or campos[1] == "" or cantidad < 0 or precio <= 0:
                    print(f"Línea {numero} inválida: valores fuera de rango.")
                    continue

                importe = cantidad * precio
                total += importe
                print(f"{campos[0]} - {campos[1]}: {cantidad} x ${precio:.2f} = ${importe:.2f}")

                if cantidad < 5:
                    pocos.append(campos[0])

        print(f"\nImporte total: ${total:.2f}")
        if pocos:
            print("Productos con cantidad menor a 5:", ", ".join(pocos))
        else:
            print("No hay productos con cantidad menor a 5.")
    except OSError as error:
        print("No se pudo leer el archivo:", error)

leer_productos()

P001 - Lapicera: 20 x $150.50 = $3010.00
P002 - Cuaderno: 3 x $1200.00 = $3600.00
P003 - Mochila: 2 x $25000.00 = $50000.00

Importe total: $56610.00
Productos con cantidad menor a 5: P002, P003


<h4>Actividad 6</h4>
<p>Los ejercicios anteriores ya manejan los errores sin usar <code>except:</code> a secas:</p>
<ul>
  <li><b>a) OSError:</b> el <code>try/except OSError</code> que rodea el <code>with open(...)</code>, al abrir, crear o escribir.</li>
  <li><b>b) ValueError:</b> el <code>try/except ValueError</code> alrededor de cada <code>int()</code> y <code>float()</code>.</li>
  <li><b>c) Registros incompletos:</b> se verifica <code>len(campos)</code> antes de convertir cualquier dato.</li>
</ul>
<p><b>try-except-finally vs. with:</b> con <code>finally</code> hay que inicializar la variable del archivo en <code>None</code> y verificarla antes de cerrar, porque la apertura podría fallar. Con <code>with</code> Python cierra el archivo solo, por eso es la opción más clara y segura.</p>
<p><code>except:</code> a secas es mala práctica porque tapa cualquier error, incluso los de nuestro propio código, y no deja ver qué falló.</p>

In [15]:
# Con try-except-finally
archivo = None                      # se inicializa porque open() podría fallar
try:
    archivo = open("productos.csv", "rt", encoding="utf-8")
    print(archivo.readline().rstrip("\n"))
except OSError as error:
    print("No se pudo acceder al archivo:", error)
finally:
    if archivo is not None:         # solo se cierra si se llegó a abrir
        archivo.close()

# Con with (más simple y seguro)
try:
    with open("productos.csv", "rt", encoding="utf-8") as archivo:
        print(archivo.readline().rstrip("\n"))
except OSError as error:
    print("No se pudo acceder al archivo:", error)

codigo;descripcion;cantidad;precio
codigo;descripcion;cantidad;precio


<h4>Actividad 7 - Desafío Proyecto Integrador</h4>
<p>Se gestionan las entregas en <code>entregas.csv</code> con los campos: número de grupo, nombre del proyecto, integrante responsable y porcentaje de avance. Cada operación recorre el archivo una sola vez y sin cargarlo completo en memoria. Todas las validaciones están en una sola función, <code>parsear_registro()</code>.</p>

In [16]:
import os

ARCHIVO = "entregas.csv"
ENCABEZADO = "grupo;proyecto;responsable;avance"


def parsear_registro(linea):
    """Devuelve (grupo, proyecto, responsable, avance) o lanza ValueError."""
    campos = linea.rstrip("\n").split(";")
    if len(campos) != 4:
        raise ValueError("cantidad de campos incorrecta")

    grupo = int(campos[0])
    if grupo <= 0:
        raise ValueError("el grupo debe ser positivo")

    proyecto = campos[1].strip()
    responsable = campos[2].strip()
    if proyecto == "" or responsable == "":
        raise ValueError("hay campos de texto vacíos")

    avance = float(campos[3].replace(",", "."))
    if not 0 <= avance <= 100:
        raise ValueError("el avance debe estar entre 0 y 100")

    return grupo, proyecto, responsable, avance


def encabezado_valido(archivo):
    return archivo.readline().rstrip("\n") == ENCABEZADO

<p><b>a) Crear el archivo con una única fila de encabezado</b> (usa "wt", así que borra lo anterior).</p>

In [17]:
def crear_archivo():
    try:
        with open(ARCHIVO, "wt", encoding="utf-8") as archivo:
            archivo.write(ENCABEZADO + "\n")
        print("Archivo creado.")
    except OSError as error:
        print("No se pudo crear el archivo:", error)

crear_archivo()

Archivo creado.


<p><b>b) Cargar entregas sin repetir el número de grupo</b></p>

In [ ]:
def grupo_existe(numero):
    """Una sola pasada: True si el grupo ya está cargado."""
    try:
        with open(ARCHIVO, "rt", encoding="utf-8") as archivo:
            if not encabezado_valido(archivo):
                return False
            for linea in archivo:
                try:
                    if parsear_registro(linea)[0] == numero:
                        return True
                except ValueError:
                    pass
    except OSError:
        return False
    return False


def guardar_entrega(grupo, proyecto, responsable, avance):
    with open(ARCHIVO, "at", encoding="utf-8") as archivo:
        archivo.write(f"{grupo};{proyecto};{responsable};{avance}\n")


def cargar_entregas():
    if not os.path.exists(ARCHIVO):
        print("Primero hay que crear el archivo.")
        return

    while True:
        try:
            grupo = int(input("Número de grupo (-1 para finalizar): "))
        except ValueError:
            print("El grupo debe ser un número entero.")
            continue
        if grupo == -1:
            break
        if grupo <= 0:
            print("El grupo debe ser positivo.")
            continue
        if grupo_existe(grupo):
            print("Ese grupo ya fue cargado.")
            continue

        proyecto = input("Nombre del proyecto: ").strip()
        responsable = input("Integrante responsable: ").strip()
        if proyecto == "" or responsable == "" or ";" in proyecto or ";" in responsable:
            print("Campos vacíos o con ';' (no permitido).")
            continue

        try:
            avance = float(input("Porcentaje de avance (0-100): ").replace(",", "."))
        except ValueError:
            print("El avance debe ser numérico.")
            continue
        if not 0 <= avance <= 100:
            print("El avance debe estar entre 0 y 100.")
            continue

        try:
            guardar_entrega(grupo, proyecto, responsable, avance)
        except OSError as error:
            print("No se pudo guardar la entrega:", error)

cargar_entregas()

Ese grupo ya fue cargado.


<p><b>c) Mostrar todas las entregas válidas</b> y <b>g)</b> informar y omitir los registros mal formados sin interrumpir el procesamiento.</p>

In [ ]:
def mostrar_entregas():
    try:
        with open(ARCHIVO, "rt", encoding="utf-8") as archivo:
            if not encabezado_valido(archivo):
                print("El archivo no tiene el encabezado esperado.")
                return
            numero = 1
            for linea in archivo:
                numero += 1
                try:
                    grupo, proyecto, responsable, avance = parsear_registro(linea)
                except ValueError as error:
                    print(f"Línea {numero} omitida: {error}")
                    continue
                print(f"Grupo {grupo} | {proyecto} | {responsable} | {avance}%")
    except OSError as error:
        print("No se pudo leer el archivo:", error)

mostrar_entregas()

Grupo 1 | Tienda Online | Ana Gomez | 80.0%
Grupo 2 | App de Turnos | Luis Fernandez | 35.0%
Grupo 3 | Gestor de Reservas | Marta Rodriguez | 45.5%


<p><b>d) Buscar una entrega por número de grupo</b> (corta apenas la encuentra).</p>

In [20]:
def buscar_entrega(numero):
    try:
        with open(ARCHIVO, "rt", encoding="utf-8") as archivo:
            if not encabezado_valido(archivo):
                print("El archivo no tiene el encabezado esperado.")
                return
            numero_linea = 1
            for linea in archivo:
                numero_linea += 1
                try:
                    grupo, proyecto, responsable, avance = parsear_registro(linea)
                except ValueError as error:
                    print(f"Línea {numero_linea} omitida: {error}")
                    continue
                if grupo == numero:
                    print(f"Grupo {grupo} | {proyecto} | {responsable} | {avance}%")
                    return
        print("No se encontró una entrega para ese grupo.")
    except OSError as error:
        print("No se pudo leer el archivo:", error)

buscar_entrega(1)

Grupo 1 | Tienda Online | Ana Gomez | 80.0%


<p><b>e) Informar los proyectos con avance inferior al 50 %</b></p>

In [21]:
def entregas_bajo_avance():
    encontrados = 0
    try:
        with open(ARCHIVO, "rt", encoding="utf-8") as archivo:
            if not encabezado_valido(archivo):
                print("El archivo no tiene el encabezado esperado.")
                return
            numero = 1
            for linea in archivo:
                numero += 1
                try:
                    grupo, proyecto, responsable, avance = parsear_registro(linea)
                except ValueError as error:
                    print(f"Línea {numero} omitida: {error}")
                    continue
                if avance < 50:
                    print(f"Grupo {grupo} | {proyecto} | {avance}%")
                    encontrados += 1
        if encontrados == 0:
            print("Ningún proyecto tiene avance inferior al 50 %.")
    except OSError as error:
        print("No se pudo leer el archivo:", error)

entregas_bajo_avance()

Grupo 2 | App de Turnos | 35.0%
Grupo 3 | Gestor de Reservas | 45.5%


<p><b>f) Calcular el promedio de avance sin dividir por cero</b></p>

In [22]:
def promedio_avance():
    suma = 0
    cantidad = 0
    try:
        with open(ARCHIVO, "rt", encoding="utf-8") as archivo:
            if not encabezado_valido(archivo):
                print("El archivo no tiene el encabezado esperado.")
                return
            numero = 1
            for linea in archivo:
                numero += 1
                try:
                    avance = parsear_registro(linea)[3]
                except ValueError as error:
                    print(f"Línea {numero} omitida: {error}")
                    continue
                suma += avance
                cantidad += 1
        if cantidad == 0:
            print("No hay registros válidos para calcular el promedio.")
        else:
            print(f"Promedio de avance: {suma / cantidad:.2f}%")
    except OSError as error:
        print("No se pudo leer el archivo:", error)

promedio_avance()

Promedio de avance: 53.50%


<p>Cada función procesa el archivo <b>una sola vez</b>, registro por registro. Los registros mal formados se informan con su número de línea y se omiten sin cortar el resto del procesamiento. El promedio verifica <code>cantidad == 0</code> antes de dividir.</p>